<a href="https://colab.research.google.com/github/YuriyaJP/20260913-Why_Employees_Leave-Measuring_Wellbeing/blob/main/20260913_Why_Leave%3DMeasuring_Wellbeing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Individual Prediction Who Is About To Leave

Dataset: IBM HR Analytics (synthetic, 1470 employees, labeled)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

df = pd.read_csv("https://raw.githubusercontent.com/IBM/employee-attrition-aif360/master/data/emp_attrition.csv")

df['Attrition_flag'] = df['Attrition'].map({'Yes': 1, 'No': 0})

cat_cols = df.select_dtypes(include='object').columns.tolist()
cat_cols.remove('Attrition')
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

drop_cols = ['Attrition', 'Attrition_flag', 'EmployeeCount', 'StandardHours', 'Over18', 'EmployeeNumber']
X = df.drop(columns=drop_cols)
y = df['Attrition_flag']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = LogisticRegression(max_iter=2000, class_weight='balanced')
model.fit(X_train, y_train)
preds = model.predict(X_test)

print("=== PART 1 RESULTS ===")
print("Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds))

coefs = pd.Series(model.coef_[0], index=X.columns).sort_values()
print("\nTop factors INCREASING attrition risk:")
print(coefs.tail(5))
print("\nTop factors DECREASING attrition risk:")
print(coefs.head(5))

=== PART 1 RESULTS ===
Accuracy: 0.7108843537414966
              precision    recall  f1-score   support

           0       0.94      0.70      0.80       247
           1       0.32      0.74      0.45        47

    accuracy                           0.71       294
   macro avg       0.63      0.72      0.63       294
weighted avg       0.84      0.71      0.75       294


Top factors INCREASING attrition risk:
BusinessTravel       0.256661
PerformanceRating    0.310414
Department           0.322000
MaritalStatus        0.531107
OverTime             0.807933
dtype: float64

Top factors DECREASING attrition risk:
EnvironmentSatisfaction   -0.359072
StockOptionLevel          -0.349356
JobSatisfaction           -0.267395
JobInvolvement            -0.251082
YearsWithCurrManager      -0.130650
dtype: float64


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


# Why People Leave Over Time

In [ ]:
"""
Generates a synthetic monthly HR tracking dataset.
One row = one employee, one month.
"""

import numpy as np
import pandas as pd

np.random.seed(42)

N_EMPLOYEES = 150
N_MONTHS = 12
DEPARTMENTS = ["Sales", "Engineering", "Customer Support", "Operations"]
START_MONTH = pd.Timestamp("2025-01-01")

rows = []

for emp_id in range(1, N_EMPLOYEES + 1):
    department = np.random.choice(DEPARTMENTS, p=[0.3, 0.3, 0.2, 0.2])
    # Each employee has some baseline "personality" - some people are just more
    # satisfied/engaged than others, independent of anything happening to them
    baseline_satisfaction = np.random.normal(3.2, 0.7)
    baseline_environment = np.random.normal(3.3, 0.7)
    years_with_manager_start = round(np.random.uniform(0, 6), 1)

    has_left = False
    leave_month = None

    for month_idx in range(N_MONTHS):
        if has_left:
            break

        month_date = START_MONTH + pd.DateOffset(months=month_idx)

        overtime = np.random.choice([0, 1], p=[0.65, 0.35])
        performance_rating = int(np.clip(np.random.normal(3.2, 0.9), 1, 5).round())
        job_satisfaction = float(np.clip(np.random.normal(baseline_satisfaction, 0.5), 1, 5).round(1))
        environment_satisfaction = float(np.clip(np.random.normal(baseline_environment, 0.5), 1, 5).round(1))
        years_with_manager = round(years_with_manager_start + month_idx / 12, 1)

        # complaints and volunteering are somewhat driven by satisfaction, plus randomness
        complaint_from_team = np.random.choice([0, 1], p=[0.9 if job_satisfaction > 2.5 else 0.75,
                                                            0.1 if job_satisfaction > 2.5 else 0.25])
        complaint_about_job = np.random.choice([0, 1], p=[0.85 if job_satisfaction > 2.5 else 0.6,
                                                            0.15 if job_satisfaction > 2.5 else 0.4])
        volunteering = np.random.choice([0, 1], p=[0.4 if job_satisfaction > 3 else 0.75,
                                                     0.6 if job_satisfaction > 3 else 0.25])

        # behavioral_change_flag: manager/HR noticed a deviation from this person's OWN baseline
        # more likely if satisfaction has dropped or complaints are stacking up
        behavior_change_prob = 0.05
        if job_satisfaction < baseline_satisfaction - 0.8:
            behavior_change_prob += 0.35
        if complaint_about_job:
            behavior_change_prob += 0.15
        behavioral_change_flag = np.random.choice([0, 1], p=[1 - behavior_change_prob, behavior_change_prob])

        # --- Build the underlying "true" leave risk this month (logistic function) ---
        risk_score = (
            -3.0
            + 0.9 * overtime
            + 0.5 * complaint_from_team
            + 0.7 * complaint_about_job
            + 0.6 * behavioral_change_flag
            - 0.5 * volunteering
            - 0.35 * (job_satisfaction - 3)
            - 0.30 * (environment_satisfaction - 3)
            - 0.12 * years_with_manager
            + 0.15 * (performance_rating - 3)  # slight positive: mirrors high-performer flight risk
        )
        leave_probability = 1 / (1 + np.exp(-risk_score))  # sigmoid -> probability between 0 and 1

        left_this_month = np.random.random() < leave_probability

        rows.append({
            "employee_id": emp_id,
            "month": month_date.strftime("%Y-%m-01"),
            "department": department,
            "overtime": overtime,
            "performance_rating": performance_rating,
            "job_satisfaction": job_satisfaction,
            "environment_satisfaction": environment_satisfaction,
            "years_with_manager": years_with_manager,
            "complaint_from_team": complaint_from_team,
            "complaint_about_job": complaint_about_job,
            "volunteering": volunteering,
            "behavioral_change_flag": behavioral_change_flag,
            "true_simulated_risk_pct": round(leave_probability * 100, 1),
            "left_company": int(left_this_month),
        })

        if left_this_month:
            has_left = True

df = pd.DataFrame(rows)
df.to_csv("hr_monthly_data.csv", index=False)
print(f"Generated {len(df)} employee-month rows for {N_EMPLOYEES} employees.")
print(f"Total departures: {df['left_company'].sum()}")
print(df.head(10).to_string())

Generated 1365 employee-month rows for 150 employees.
Total departures: 68
   employee_id       month   department  overtime  performance_rating  job_satisfaction  environment_satisfaction  years_with_manager  complaint_from_team  complaint_about_job  volunteering  behavioral_change_flag  true_simulated_risk_pct  left_company
0            1  2025-01-01  Engineering         0                   3               2.9                       3.2                 0.9                    0                    0             0                       0                      4.2             0
1            1  2025-02-01  Engineering         0                   3               2.2                       3.2                 1.0                    0                    0             1                       0                      3.2             0
2            1  2025-03-01  Engineering         0                   3               2.2                       3.2                 1.1                    0            

In [ ]:
"""
Trains a logistic regression model on ONLY the columns a real HR team could
actually track (no cheating with the hidden 'true_simulated_risk_pct' column,
which only exists because we generated fake data and needs to be dropped here
so the exercise mirrors reality).

Then:
 1. Shows which factors the model actually picked up on (compare to what we
    built into the simulation, as a sanity check).
 2. Adds the model's own predicted_risk_pct back onto the dataset.
 3. For every employee who left, shows the risk % the model gave them the
    month right before they left -- this is the "confirmation" data point:
    what threshold did real leavers cross, so you know what to watch for
    with people still on your team.
"""

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import LabelEncoder

df = pd.read_csv("hr_monthly_data.csv")

le = LabelEncoder()
df["department_encoded"] = le.fit_transform(df["department"])

FEATURES = [
    "overtime", "performance_rating", "job_satisfaction", "environment_satisfaction",
    "years_with_manager", "complaint_from_team", "complaint_about_job",
    "volunteering", "behavioral_change_flag", "department_encoded",
]
TARGET = "left_company"

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = LogisticRegression(max_iter=2000, class_weight="balanced")
model.fit(X_train, y_train)
preds = model.predict(X_test)

print("=== MODEL PERFORMANCE ===")
print("Accuracy:", round(accuracy_score(y_test, preds), 3))
print(classification_report(y_test, preds))

print("\n=== WHAT THE MODEL LEARNED (coefficients) ===")
coefs = pd.Series(model.coef_[0], index=FEATURES).sort_values()
print(coefs)

# --- Add the model's own predictions back onto the full dataset ---
df["predicted_risk_pct"] = (model.predict_proba(X[FEATURES])[:, 1] * 100).round(1)
df.to_csv("hr_monthly_data_with_predictions.csv", index=False)

leavers = df[df["left_company"] == 1].copy()
print(f"\n=== {len(leavers)} DEPARTURES: model's predicted risk %, the month they left ===")
print(leavers[["employee_id", "month", "department", "predicted_risk_pct"]]
      .sort_values("predicted_risk_pct", ascending=False)
      .to_string(index=False))

print("\nAverage predicted risk % at time of leaving:", round(leavers["predicted_risk_pct"].mean(), 1))
print("Lowest predicted risk % among people who still left:", leavers["predicted_risk_pct"].min())

=== MODEL PERFORMANCE ===
Accuracy: 0.667
              precision    recall  f1-score   support

           0       0.97      0.67      0.79       259
           1       0.09      0.57      0.15        14

    accuracy                           0.67       273
   macro avg       0.53      0.62      0.47       273
weighted avg       0.92      0.67      0.76       273


=== WHAT THE MODEL LEARNED (coefficients) ===
environment_satisfaction   -0.483175
years_with_manager         -0.371738
volunteering               -0.306340
job_satisfaction           -0.282997
department_encoded         -0.089399
performance_rating          0.312979
behavioral_change_flag      0.327947
complaint_from_team         0.509754
overtime                    0.529774
complaint_about_job         1.100595
dtype: float64

=== 68 DEPARTURES: model's predicted risk %, the month they left ===
 employee_id      month       department  predicted_risk_pct
         139 2025-04-01      Engineering                94.5
       